# Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 17.05.2024
- Date of Last Modification: 12.08.2025

## 1. Setup

### 1.1 Import Libraries

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import json
import os
import pickle
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import scipy.sparse as sp
#import scgpt as scg
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from pyensembl import EnsemblRelease
from scipy.stats import spearmanr
#from scgpt.model import TransformerModel
#from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils

### 1.2 Run Notebook Setup

In [9]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 1.3 Configure Paths and Create Directories

In [10]:
# Define paths
st_data_folder_path = "../../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
sc_data_folder_path = "../../datasets/sc_data" # spatially resolved transcriptomics data
sc_data_bronze_folder_path = f"{sc_data_folder_path}/bronze"
sc_data_silver_folder_path = f"{sc_data_folder_path}/silver"
sc_data_gold_folder_path = f"{sc_data_folder_path}/gold"
tokenizer = "cell_graph_tokenizer"
tokenizer_folder_path = f"../../datasets/tokenizer/{tokenizer}"
results_folder_path = "../../datasets/results"
geneformer_token_dictionary_file = f"../../datasets/tokenizer/geneformer_token_dictionary.pkl"
scgpt_model_dir = Path("../../artifacts/scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../../artifacts/scgpt/metadata/gene_info.csv")

## 2. Tokenization

In [11]:
tokenizer = 'cell-graph-tokenizer'

In [ ]:
# Load token dict
print(f"Loading token dictionary from {token_dictionary_file_path}")
with open(token_dictionary_file_path, "rb") as file:
    token_dict = pickle.load(file)
print(f"Number of Gene Tokens: {sum('ENS' in key for key in token_dict.keys())}.")
print(f"Number of Tokens: {len(token_dict)}.")

In [ ]:
# Initialize tokenizer
if tokenizer == 'cell-neighborhood-tokenizer':
    Tokenizer = CellNeighborhoodTokenizer
elif tokenizer == 'cell-graph-tokenizer':
    Tokenizer = CellGraphTokenizer
tk = Tokenizer(
    nproc=nproc,
    processing_mode=processing_mode,
    model_input_size=model_input_size,
    include_zero_expr_genes=include_zero_expr_genes,
    n_neighs=n_neighs,
    radius=radius,
    delaunay=delaunay,
    rank_cell_norm_method=rank_cell_norm_method,
    rank_gene_norm_method=rank_gene_norm_method,
    rank_count_norm_method=rank_count_norm_method,
    count_cell_norm_method=count_cell_norm_method,
    count_gene_norm_method=count_gene_norm_method,
    count_count_norm_method=count_count_norm_method,
    norm_factor_file_path=norm_factor_file_path,
    token_dictionary_file_path=token_dictionary_file_path)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                 output_directory=output_directory,
                 output_file_prefix=output_file_prefix,
                 use_generator=use_generator,
                 cache_directory_path=cache_directory_path,
                 keep_in_memory=keep_in_memory,
                 num_shards=num_shards)

In [ ]:
tokens_per_cell = 32
n_neighbors = 10
model_input_size = (n_neighbors + 1) * tokens_per_cell
norm_method = 'shifted_log'
norm_factor = None

for norm_factor in norm_factors:
    output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"

    tk = tokenizers.CellGraphTokenizer(
            model_input_size=model_input_size,
            tokens_per_cell=tokens_per_cell,
            norm_method=norm_method,
            norm_factor=norm_factor,
            norm_factor_file_path=norm_factor_file_path,
            token_dictionary_file_path=token_dictionary_file_path,
    )
        
    tk.tokenize_data(input_directory=input_directory,
                     output_directory=output_directory,
                     output_file_prefix=output_file_prefix)